In [0]:
netflix_data = spark.table("workspace.default.netflix_titles")
#  schema and print first rows:       
netflix_data.printSchema()
netflix_data.show(3)

root
 |-- show_id: string (nullable = true)
 |-- type: string (nullable = true)
 |-- title: string (nullable = true)
 |-- director: string (nullable = true)
 |-- cast: string (nullable = true)
 |-- country: string (nullable = true)
 |-- date_added: string (nullable = true)
 |-- release_year: long (nullable = true)
 |-- rating: string (nullable = true)
 |-- duration: string (nullable = true)
 |-- listed_in: string (nullable = true)
 |-- description: string (nullable = true)

+-------+-------+--------------------+---------------+--------------------+-------------+------------------+------------+------+---------+--------------------+--------------------+
|show_id|   type|               title|       director|                cast|      country|        date_added|release_year|rating| duration|           listed_in|         description|
+-------+-------+--------------------+---------------+--------------------+-------------+------------------+------------+------+---------+--------------------+

In [0]:
netflix_data.count()

8809

1. Count of Movies vs TV Shows.

In [0]:
netflix_data.groupBy("type").count().show()

+-------------+-----+
|         type|count|
+-------------+-----+
|        Movie| 6131|
|      TV Show| 2676|
|         NULL|    1|
|William Wyler|    1|
+-------------+-----+



2. Count of null/missing director values.

In [0]:

netflix_data.filter(netflix_data.director.isNull()).count()


2636

3. Count of titles per rating, sorted descending.

In [0]:
netflix_data.groupBy("rating").count().orderBy("count", ascending=False).show()

+--------------------+-----+
|              rating|count|
+--------------------+-----+
|               TV-MA| 3207|
|               TV-14| 2160|
|               TV-PG|  862|
|                   R|  799|
|               PG-13|  490|
|               TV-Y7|  334|
|                TV-Y|  307|
|                  PG|  287|
|                TV-G|  220|
|                  NR|   80|
|                   G|   41|
|            TV-Y7-FV|    6|
|                NULL|    6|
|               NC-17|    3|
|                  UR|    3|
|              74 min|    1|
|              84 min|    1|
|              66 min|    1|
|Classic Movies, D...|    1|
+--------------------+-----+



4. Top 10 listed_in genres by title count


In [0]:
netflix_data.groupBy("listed_in").count().orderBy("count", ascending=False).show(10, truncate= False)

+------------------------------------------------+-----+
|listed_in                                       |count|
+------------------------------------------------+-----+
|Dramas, International Movies                    |362  |
|Documentaries                                   |359  |
|Stand-Up Comedy                                 |334  |
|Comedies, Dramas, International Movies          |274  |
|Dramas, Independent Movies, International Movies|252  |
|Kids' TV                                        |220  |
|Children & Family Movies                        |215  |
|Children & Family Movies, Comedies              |201  |
|Documentaries, International Movies             |186  |
|Dramas, International Movies, Romantic Movies   |180  |
+------------------------------------------------+-----+
only showing top 10 rows


5. Top 15 genres by title count

In [0]:
from pyspark.sql.functions import split, explode, trim, col

netflix_data \
    .withColumn("genre", explode(split(col("listed_in"), ","))) \
    .withColumn("genre", trim(col("genre"))) \
    .groupBy("genre") \
    .count() \
    .orderBy("count", ascending=False) \
    .show(15, truncate=False)

+------------------------+-----+
|genre                   |count|
+------------------------+-----+
|International Movies    |2752 |
|Dramas                  |2427 |
|Comedies                |1674 |
|International TV Shows  |1351 |
|Documentaries           |868  |
|Action & Adventure      |859  |
|TV Dramas               |763  |
|Independent Movies      |756  |
|Children & Family Movies|641  |
|Romantic Movies         |616  |
|TV Comedies             |581  |
|Thrillers               |577  |
|Crime TV Shows          |470  |
|Kids' TV                |451  |
|Docuseries              |395  |
+------------------------+-----+
only showing top 15 rows


6. Type of available content

In [0]:
netflix_data.filter(netflix_data.date_added.isNull()).count()

12

7. Quantity of newly added contents by month and year

In [0]:
from pyspark.sql.functions import col, try_to_date, year, month, count

df = netflix_data.withColumn(
    "added_date",
    try_to_date(trim(col("date_added")), "MMMM d, yyyy")
)

df = df.withColumn("year", year(col("added_date"))) \
       .withColumn("month", month(col("added_date")))

df.filter(col("added_date").isNotNull()) \
  .groupBy("year", "month") \
  .count() \
  .orderBy("year", "month") \
  .show(50)


+----+-----+-----+
|year|month|count|
+----+-----+-----+
|2008|    1|    1|
|2008|    2|    1|
|2009|    5|    1|
|2009|   11|    1|
|2010|   11|    1|
|2011|    5|    1|
|2011|    9|    1|
|2011|   10|   11|
|2012|    2|    1|
|2012|   11|    1|
|2012|   12|    1|
|2013|    3|    1|
|2013|    8|    1|
|2013|    9|    2|
|2013|   10|    3|
|2013|   11|    2|
|2013|   12|    2|
|2014|    1|    2|
|2014|    2|    2|
|2014|    4|    2|
|2014|    6|    1|
|2014|    7|    1|
|2014|    8|    1|
|2014|    9|    1|
|2014|   10|    4|
|2014|   11|    4|
|2014|   12|    6|
|2015|    1|    1|
|2015|    2|    4|
|2015|    3|    5|
|2015|    4|    5|
|2015|    5|    6|
|2015|    6|    6|
|2015|    7|    7|
|2015|    8|    2|
|2015|    9|    7|
|2015|   10|   14|
|2015|   11|    4|
|2015|   12|   21|
|2016|    1|   43|
|2016|    2|   15|
|2016|    3|   17|
|2016|    4|   22|
|2016|    5|   13|
|2016|    6|   18|
|2016|    7|   28|
|2016|    8|   34|
|2016|    9|   48|
|2016|   10|   51|
|2016|   11|

In [0]:
from pyspark.sql.functions import col

netflix_data.filter(
    col("date_added").contains("TV-PG")
).select(
    "show_id", "title", "type",
    "director", "rating", "date_added"
).show(truncate=False)

+----------------+-----+-------------+-------------+-----------------------------+----------+
|show_id         |title|type         |director     |rating                       |date_added|
+----------------+-----+-------------+-------------+-----------------------------+----------+
|Flying Fortress"|NULL |William Wyler|United States|Classic Movies, Documentaries|TV-PG     |
+----------------+-----+-------------+-------------+-----------------------------+----------+



8. Average duration of Tv Shows in seasons

In [0]:
from pyspark.sql.functions import col, split, avg, round

result = (
    netflix_data
    .filter(col("type") == "TV Show")
    .filter(col("duration").like("%Season%"))
    .withColumn(
        "seasons",
        split(col("duration"), " ")[0].cast("int")
    )
    .agg(
        round(avg("seasons"), 2).alias("average_seasons")
    )
)

result.show()

+---------------+
|average_seasons|
+---------------+
|           1.76|
+---------------+



9. Its Christmas! What movies/Tv shows are you binge watching this weekend? 

In [0]:
from pyspark.sql.functions import col

netflix_data.filter(
    (col("title").like("%Christmas%")) |
    (col("description").like("%Christmas%"))
).select("show_id", "title", "type", "description").show(truncate=False)

+-------+--------------------------------------+-------+---------------------------------------------------------------------------------------------------------------------------------------------------------+
|show_id|title                                 |type   |description                                                                                                                                              |
+-------+--------------------------------------+-------+---------------------------------------------------------------------------------------------------------------------------------------------------------+
|s657   |Love Is a Story                       |Movie  |Two young artists, a violin player and an actress, fight to achieve their dreams and keep their relationship together during Christmastime.              |
|s1517  |Home for Christmas                    |TV Show|Tired of the constant comments on her relationship status, perpetually single Johanne starts a 24-da

10. My friend is a    Ryan Reynold fan , I am thinking of making a movie compilation. What movies should I include ?

In [0]:
from pyspark.sql.functions import col

netflix_data.filter(
    (col("cast").like("%Ryan Reynolds%"))
).select("show_id", "title", "type").show(truncate=False)

+-------+-------------------------------------+-----+
|show_id|title                                |type |
+-------+-------------------------------------+-----+
|s47    |Safe House                           |Movie|
|s144   |Green Lantern                        |Movie|
|s3032  |Betty White: First Lady of Television|Movie|
|s3151  |6 Underground                        |Movie|
|s3583  |Selfless                             |Movie|
|s6929  |Harold & Kumar Go to White Castle    |Movie|
|s7153  |Just Friends                         |Movie|
|s7475  |Mississippi Grind                    |Movie|
|s8189  |The Amityville Horror                |Movie|
|s8237  |The Captive                          |Movie|
|s8259  |The Croods                           |Movie|
|s8642  |Turbo                                |Movie|
|s8690  |Waiting...                           |Movie|
+-------+-------------------------------------+-----+



11. How many movies or shows were released during COVID season.

In [0]:
from pyspark.sql.functions import col

netflix_data.filter(
    (col("release_year")>= 2019) & (col("release_year")<= 2023)
).groupBy("release_year").count().orderBy("release_year").show()

+------------+-----+
|release_year|count|
+------------+-----+
|        2019| 1030|
|        2020|  953|
|        2021|  592|
+------------+-----+



12. I am on a short flight of 2 hours, which movies or shows I should download to watch on the flight ?


In [0]:
from pyspark.sql.functions import col, split

result = (
    netflix_data
    .withColumn("duration_num", split(col("duration"), " ")[0].try_cast("int"))
    .filter(
        (
            (col("type") == "Movie") &
            (col("duration").like("%min%")) &
            (col("duration_num") <= 120)
        )
        |
        (
            (col("type") == "TV Show") &
            (col("duration").like("%min%")) &
            (col("duration_num") <= 120)
        )
    )
    .select("title", "type", "duration")
)

#display(result)
result.show(10)

+--------------------+-----+--------+
|               title| type|duration|
+--------------------+-----+--------+
|Dick Johnson Is Dead|Movie|  90 min|
|My Little Pony: A...|Movie|  91 min|
|        The Starling|Movie| 104 min|
|Confessions of an...|Movie|  91 min|
|Europe's Most Dan...|Movie|  67 min|
|           Intrusion|Movie|  94 min|
|Go! Go! Cory Cars...|Movie|  61 min|
|           Grown Ups|Movie| 103 min|
|          Dark Skies|Movie|  97 min|
|            Paranoia|Movie| 106 min|
+--------------------+-----+--------+
only showing top 10 rows
